# Stage 1: Data Preparation

DVC stage `prepare`. Reads the raw `hotel_bookings.csv`, cleans it, engineers features, splits it into train and test sets and encodes categorical columns.

All settings come from the `prepare` section of `params.yaml`, so changing a value there makes `dvc repro` rerun this stage and every stage after it.

In [ ]:
import json
from pathlib import Path

import joblib
import numpy as np
import pandas as pd
import yaml
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OrdinalEncoder

## Load parameters

In [ ]:
with open('params.yaml') as f:
    params = yaml.safe_load(f)

seed = params['base']['random_state']
data_cfg = params['data']
prep_cfg = params['prepare']
processed_dir = Path(data_cfg['processed_dir'])
processed_dir.mkdir(parents=True, exist_ok=True)
Path('models').mkdir(exist_ok=True)
prep_cfg

## Load raw data

In [ ]:
df = pd.read_csv(data_cfg['raw'])
print(f'Raw shape: {df.shape}')
df.head()

## Cleaning

Same rules as the exploration notebook: fill missing values, remove duplicate bookings, drop personal data columns and drop `reservation_status` / `reservation_status_date`, which are only known after the booking outcome and would leak the target.

In [ ]:
df['children'] = df['children'].fillna(0)
df['country'] = df['country'].fillna('Unknown')
df['agent'] = df['agent'].fillna(0)
df['company'] = df['company'].fillna(0)

df = df.drop_duplicates()
df = df.drop(columns=prep_cfg['drop_columns'], errors='ignore')
print(f'Shape after cleaning: {df.shape}')
print(f'Missing values left: {int(df.isnull().sum().sum())}')

## Feature engineering

In [ ]:
month_map = {'January': 1, 'February': 2, 'March': 3, 'April': 4, 'May': 5, 'June': 6,
             'July': 7, 'August': 8, 'September': 9, 'October': 10, 'November': 11, 'December': 12}
df['arrival_date_month'] = df['arrival_date_month'].map(month_map)
df['Total_Guests'] = df['adults'] + df['children'] + df['babies']
df['Stay_Duration'] = df['stays_in_weekend_nights'] + df['stays_in_week_nights']

df = df[(df['adr'] >= 0) & (df['Total_Guests'] > 0)].reset_index(drop=True)
print(f'Shape after removing invalid rows: {df.shape}')
print(f"Cancellation rate: {df['is_canceled'].mean():.3f}")

## Train / test split

Stratified on the target so both sets keep the same cancellation rate.

In [ ]:
train_df, test_df = train_test_split(
    df,
    test_size=prep_cfg['test_size'],
    random_state=seed,
    stratify=df['is_canceled'],
)
train_df = train_df.reset_index(drop=True)
test_df = test_df.reset_index(drop=True)
print(f'Train: {train_df.shape}, Test: {test_df.shape}')

## Outlier clipping

IQR bounds are learned on the training set only and then applied to both sets, so no information from the test set leaks into preprocessing.

In [ ]:
clip_bounds = {}
for col in prep_cfg['clip_columns']:
    q1, q3 = train_df[col].quantile([0.25, 0.75])
    iqr = q3 - q1
    low, high = float(q1 - prep_cfg['iqr_multiplier'] * iqr), float(q3 + prep_cfg['iqr_multiplier'] * iqr)
    clip_bounds[col] = (low, high)
    train_df[col] = train_df[col].clip(low, high)
    test_df[col] = test_df[col].clip(low, high)
    print(f'{col}: clipped to [{low:.1f}, {high:.1f}]')

## Encode categorical columns

The encoder is fitted on the training set. Categories that only appear in the test set are mapped to `-1`.

In [ ]:
cat_cols = [c for c in train_df.columns if not pd.api.types.is_numeric_dtype(train_df[c])]
encoder = OrdinalEncoder(handle_unknown='use_encoded_value', unknown_value=-1)
train_df[cat_cols] = encoder.fit_transform(train_df[cat_cols].astype(str))
test_df[cat_cols] = encoder.transform(test_df[cat_cols].astype(str))
print('Encoded columns:', cat_cols)

## Save outputs

In [ ]:
train_df.to_csv(processed_dir / 'train.csv', index=False)
test_df.to_csv(processed_dir / 'test.csv', index=False)
joblib.dump({'encoder': encoder, 'categorical_columns': cat_cols, 'clip_bounds': clip_bounds},
            'models/preprocessor.pkl')

print(json.dumps({
    'train_rows': len(train_df),
    'test_rows': len(test_df),
    'features': train_df.shape[1] - 1,
    'train_cancel_rate': round(float(train_df['is_canceled'].mean()), 4),
    'test_cancel_rate': round(float(test_df['is_canceled'].mean()), 4),
}, indent=2))